In [1]:
import matplotlib

matplotlib.use("Agg")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

PLOT_DIR = Path("_plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# Full hallway run 2026-05-14 (hallway_full). Prefer latest log; fall back to older captures.
for _name in (
    "hallway_full_20260514_060054.csv",
    "hallway_straight_20260514_054641.csv",
    "hallway_225725.csv",
):
    _csv = Path(_name)
    if _csv.is_file():
        break
else:
    raise FileNotFoundError("No CSV found beside this notebook.")
df = pd.read_csv(_csv)
print("Loaded:", _csv.resolve(), "rows=", len(df))

Loaded: /Users/naqotrunnadda/Documents/roboracer-t7/test_manual_map_logger/hallway_full_20260514_060054.csv rows= 1546


In [3]:
df

,time_sec,frame_id,x,y,z,yaw_rad,left_wall_m,right_wall_m,scan_stamp_sec
0,1.778738e+09,odom,0.000000,0.000000,0.0,0.000000,0.843192,0.389740,1.778738e+09
1,1.778738e+09,odom,6.593549,3.179852,0.0,0.850485,0.843192,0.389740,1.778738e+09
2,1.778738e+09,odom,0.000000,0.000000,0.0,0.000000,0.843192,0.389740,1.778738e+09
3,1.778738e+09,odom,0.000000,0.000000,0.0,0.000000,0.846038,0.403352,1.778738e+09
4,1.778738e+09,odom,0.000000,0.000000,0.0,0.000000,0.843948,0.408593,1.778738e+09
...,...,...,...,...,...,...,...,...,...
1541,1.778739e+09,odom,3.340580,7.192562,0.0,3.107304,0.774956,0.698584,1.778739e+09
1542,1.778739e+09,odom,3.429079,7.249798,0.0,3.059946,0.767866,0.672657,1.778739e+09
1543,1.778739e+09,odom,3.429079,7.249798,0.0,3.059946,0.767866,0.672657,1.778739e+09
1544,1.778739e+09,odom,3.429079,7.249798,0.0,3.059946,0.767225,0.692298,1.778739e+09


In [4]:
df.yaw_rad.describe()

count    1546.000000
mean        0.844728
std         1.702377
min        -3.111036
25%        -0.089679
50%         1.415484
75%         2.051948
max         3.141105
Name: yaw_rad, dtype: float64

In [5]:
df = df[['x', 'y', 'right_wall_m', 'left_wall_m']]
df = df.rename(columns={'x': 'x_m', 'y': 'y_m', 'right_wall_m': 'w_tr_right_m', 'left_wall_m': 'w_tr_left_m'})

In [6]:
# Remove duplicate centerline points
# df = df.drop_duplicates(subset=["x_m", "y_m"])

x = df["x_m"].to_numpy()
y = df["y_m"].to_numpy()

w_right = df["w_tr_right_m"].to_numpy()
w_left  = df["w_tr_left_m"].to_numpy()

# Compute tangents
dx = np.gradient(x)
dy = np.gradient(y)

length = np.sqrt(dx**2 + dy**2)

# Prevent divide-by-zero
eps = 1e-9
length[length < eps] = eps

# Unit normals
nx = -dy / length
ny = dx / length

# Boundaries
left_x  = x + nx * w_left
left_y  = y + ny * w_left

right_x = x - nx * w_right
right_y = y - ny * w_right

# Plot
plt.figure(figsize=(8,8))

plt.plot(x, y, 'k--', linewidth=1, label="Centerline")
plt.plot(left_x, left_y, 'b', label="Left")
plt.plot(right_x, right_y, 'r', label="Right")

plt.axis("equal")
plt.legend()
plt.savefig(PLOT_DIR / "hallway_full_centerline_walls.png", dpi=150, bbox_inches="tight")
plt.close()

In [7]:
df['wall_side_len'] = df['w_tr_right_m'] + df['w_tr_left_m']
df['wall_side_len'].describe()

count    1546.000000
mean        2.086953
std         0.670399
min         1.043428
25%         1.492803
50%         2.020941
75%         2.382001
max         3.475450
Name: wall_side_len, dtype: float64

In [8]:
df["t"] = range(len(df))

# Plot x over time/index
plt.figure(figsize=(10,4))

plt.plot(df["t"], df["x_m"])

plt.xlabel("Chronological Index")
plt.ylabel("x_m")
plt.title("x_m Over Track Sequence")

plt.grid(True)
plt.savefig(PLOT_DIR / "hallway_full_x_over_index.png", dpi=150, bbox_inches="tight")
plt.close()

In [9]:
plt.plot(df["t"], df["y_m"])

plt.xlabel("Chronological Index")
plt.ylabel("y_m")
plt.title("y_m Over Track Sequence")

plt.grid(True)
plt.savefig(PLOT_DIR / "hallway_full_y_over_index.png", dpi=150, bbox_inches="tight")
plt.close()

### Run analysis (hallway_full)

This section summarizes whether the logged `odom` → `base_link` pose is **physically consistent** (smooth motion) or **dominated by estimator / driver glitches**.

**Healthy hall pass:** integrated path length ≈ few–tens of meters, per-step displacement mostly under a few cm at 20 Hz, yaw changes gradual.

**Broken stack (common in this log):** `vesc_driver_node` **SIGSEGV (-11)** → no reliable wheel odometry → EKF starved / jumping → TF snaps between states. The logger is **correct**; it records the broken tree.

In [10]:
import numpy as np

raw = pd.read_csv(_csv)
x, y = raw["x"].to_numpy(), raw["y"].to_numpy()
step = np.hypot(np.diff(x), np.diff(y))
dur = float(raw["time_sec"].iloc[-1] - raw["time_sec"].iloc[0])
path_len = float(step.sum())
med = float(np.median(step)) if len(step) else 0.0
p95 = float(np.percentile(step, 95)) if len(step) else 0.0
p99 = float(np.percentile(step, 99)) if len(step) else 0.0
big = int((step > 0.5).sum())  # >0.5 m between 20 Hz samples => teleport

print("--- Diagnostics:", _csv.name, "---")
print(f"rows={len(raw)}  duration_s={dur:.2f}  mean_dt_ms={1000*dur/max(len(raw)-1,1):.1f}")
print(f"path_length_sum(|Δ|)={path_len:.2f} m  (if >> hall length, TF is jumping)")
print(f"step_m: median={med:.4f}  p95={p95:.4f}  p99={p99:.4f}  jumps>0.5m: {big}")
print(f"x range [{x.min():.3f}, {x.max():.3f}]  y range [{y.min():.3f}, {y.max():.3f}]")
dyaw = (raw["yaw_rad"].iloc[-1] - raw["yaw_rad"].iloc[0]) * 57.2958
print(f"Δyaw start→end (deg): {dyaw:.1f}")


--- Diagnostics: hallway_full_20260514_060054.csv ---
rows=1546  duration_s=79.46  mean_dt_ms=51.4
path_length_sum(|Δ|)=2392.09 m  (if >> hall length, TF is jumping)
step_m: median=0.0183  p95=6.9377  p99=7.4576  jumps>0.5m: 586
x range [-3.286, 6.763]  y range [-0.190, 7.250]
Δyaw start→end (deg): 175.3
